# Carga de diagramas do dataset do Kaggle e análise STRIDE (v6)

**v6:** exporta o relatório STRIDE em **PDF** e **DOCX** no Google Drive. Mantém leitura de todos os `.png` em `diagramas_aprovados/`, verificação local e download condicional do dataset.

In [ ]:
# ==========================================
# ETAPA 0: AUTORIZAÇÃO E PASTA NO GOOGLE DRIVE
# ==========================================
# Todos os arquivos baixados e gerados neste notebook são gravados no Google Drive.
# Execute esta célula primeiro e autorize o acesso à sua conta Google quando solicitado.

from pathlib import Path

from google.colab import drive

print("Solicitando autorização para acessar o Google Drive...")
print("Clique no link exibido, escolha a conta Google e cole o código de autorização se solicitado.\n")

drive.mount("/content/drive", force_remount=False)

# Nome da pasta raiz do projeto no Google Drive (My Drive)
PROJECT_FOLDER_NAME = "TechChallenger_STRIDE"

DRIVE_ROOT = Path("/content/drive/MyDrive") / PROJECT_FOLDER_NAME
DATASET_AUGMENTED_DIR = DRIVE_ROOT / "dataset_augmented"
OUTPUT_DIR = DRIVE_ROOT / "diagramas_aprovados"
REPORTS_DIR = DRIVE_ROOT / "relatorios"

for folder in (DRIVE_ROOT, DATASET_AUGMENTED_DIR, OUTPUT_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Pastas mapeadas no Google Drive:")
print(f"  Raiz do projeto    : {DRIVE_ROOT}")
print(f"  Dataset (augmented) : {DATASET_AUGMENTED_DIR}")
print(f"  Diagramas           : {OUTPUT_DIR}")
print(f"  Relatórios/PDFs     : {REPORTS_DIR}")


def verificar_pastas_locais(pastas):
    """Busca e resume pastas e arquivos locais no Google Drive."""
    print("\nVerificando pastas e arquivos locais:")
    resumo = {}
    extensoes_imagem = {".png", ".jpg", ".jpeg"}

    for nome, pasta in pastas.items():
        pasta = Path(pasta)
        if not pasta.exists():
            print(f"  [AUSENTE] {nome}: {pasta}")
            resumo[nome] = {"existe": False, "arquivos": 0, "imagens": 0}
            continue

        arquivos = [f for f in pasta.rglob("*") if f.is_file()]
        imagens = [f for f in arquivos if f.suffix.lower() in extensoes_imagem]
        print(f"  [OK] {nome}: {pasta}")
        print(f"       arquivos: {len(arquivos)} | imagens: {len(imagens)}")
        resumo[nome] = {
            "existe": True,
            "arquivos": len(arquivos),
            "imagens": len(imagens),
        }

    return resumo


PASTAS_PROJETO = {
    "Raiz do projeto": DRIVE_ROOT,
    "Dataset (augmented)": DATASET_AUGMENTED_DIR,
    "Diagramas aprovados": OUTPUT_DIR,
    "Relatórios/PDFs": REPORTS_DIR,
}
STATUS_PASTAS = verificar_pastas_locais(PASTAS_PROJETO)

print("\nAutorização concluída. Prossiga para a próxima célula.")


In [ ]:
# ==========================================
# ETAPA 1: PREPARAÇÃO E DOWNLOAD DO DATASET
# ==========================================
import os
import json
import cv2
import shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import userdata

# Instala bibliotecas necessárias para a IA de classificação visual
!pip install -q transformers torch kaggle

from transformers import pipeline
from PIL import Image

# Usa as pastas definidas na etapa anterior (Google Drive)
DATASET_AUGMENTED_DIR = str(DATASET_AUGMENTED_DIR)
OUTPUT_DIR = str(OUTPUT_DIR)
REPORTS_DIR = str(REPORTS_DIR)


def dataset_local_disponivel(diretorio):
    """Retorna True se já existirem imagens do dataset no diretório local."""
    path = Path(diretorio)
    if not path.exists():
        return False

    extensoes_imagem = {".png", ".jpg", ".jpeg"}
    for _, _, files in os.walk(path):
        for file in files:
            if Path(file).suffix.lower() in extensoes_imagem:
                return True
    return False


# ==========================================
# ETAPA 1: VERIFICAR DATASET LOCAL / DOWNLOAD
# ==========================================
print("Verificando se o dataset já existe localmente no Google Drive...")
dataset_ja_existe = dataset_local_disponivel(DATASET_AUGMENTED_DIR)

if dataset_ja_existe:
    print(f"Dataset encontrado em: {DATASET_AUGMENTED_DIR}")
    print("Download do Kaggle ignorado.")
else:
    print("Dataset não encontrado. Configurando credenciais do Kaggle...")
    try:
        kaggle_secret = userdata.get("my-kaggle")
        kaggle_creds = json.loads(kaggle_secret)
        os.environ["KAGGLE_USERNAME"] = kaggle_creds["username"]
        os.environ["KAGGLE_KEY"] = kaggle_creds["key"]
    except Exception as e:
        print("ERRO: Verifique se o secret 'my-kaggle' contém um JSON válido com 'username' e 'key'.")
        raise e

    print("Baixando o dataset do Kaggle para o Google Drive...")
    print(f"Destino: {DATASET_AUGMENTED_DIR}")
    !kaggle datasets download -d carlosrian/software-architecture-dataset --unzip -p "{DATASET_AUGMENTED_DIR}"

    if not dataset_local_disponivel(DATASET_AUGMENTED_DIR):
        raise FileNotFoundError(
            f"Download concluído, mas nenhuma imagem foi encontrada em {DATASET_AUGMENTED_DIR}."
        )
    print("Download concluído com sucesso.")

# ==========================================
# ETAPA 1b: REMOVER ARQUIVOS .XML
# ==========================================
def remover_arquivos_xml(diretorio):
    """Remove recursivamente todos os arquivos .xml do diretório informado."""
    removidos = 0
    for root, _, files in os.walk(diretorio):
        for file in files:
            if file.lower().endswith(".xml"):
                os.remove(os.path.join(root, file))
                removidos += 1
    return removidos

print("Removendo arquivos .xml do dataset...")
total_xml_removidos = remover_arquivos_xml(DATASET_AUGMENTED_DIR)
print(f"{total_xml_removidos} arquivo(s) .xml removido(s) de {DATASET_AUGMENTED_DIR}")

# ==========================================
# ETAPA 2: FUNÇÕES DE ANÁLISE (CRITÉRIOS)
# ==========================================
print("Carregando modelo de IA (CLIP) para análise semântica...")
image_classifier = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

def is_sharp(image_cv, threshold=100.0):
    """Verifica se a imagem é nítida usando a variância do Laplaciano."""
    gray = cv2.cvtColor(image_cv, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    return variance > threshold

def is_aligned(image_cv):
    """Verifica se a imagem não está inclinada usando a Transformada de Hough."""
    gray = cv2.cvtColor(image_cv, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, 200)

    if lines is None:
        return True

    skewed_lines = 0
    for line in lines:
        rho, theta = line[0]
        angle = np.degrees(theta)
        if not ((angle < 5 or angle > 175) or (85 < angle < 95)):
            skewed_lines += 1

    return skewed_lines < 5

def is_architecture_diagram(image_path):
    """Usa IA para verificar se é um diagrama de arquitetura e se parece inteiro."""
    try:
        image = Image.open(image_path)
        labels = [
            "a complete software architecture diagram",
            "a cropped or incomplete image",
            "a random photo",
            "a text document",
        ]
        results = image_classifier(image, candidate_labels=labels)
        best_match = results[0]["label"]
        return best_match == "a complete software architecture diagram"
    except Exception:
        return False

# ==========================================
# ETAPAS 3 e 4: EXAMINAR, FILTRAR E SALVAR
# ==========================================
print("Iniciando a filtragem das imagens. Isso pode levar alguns minutos...")
aprovadas = []
max_imagens = 10

for root, _, files in os.walk(DATASET_AUGMENTED_DIR):
    for file in files:
        if len(aprovadas) >= max_imagens:
            break

        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            file_path = os.path.join(root, file)
            img_cv = cv2.imread(file_path)
            if img_cv is None:
                continue

            if not is_sharp(img_cv, threshold=150.0):
                continue
            if not is_aligned(img_cv):
                continue
            if not is_architecture_diagram(file_path):
                continue

            dest_path = os.path.join(OUTPUT_DIR, file)
            shutil.copy(file_path, dest_path)
            aprovadas.append(dest_path)

print(f"\n{len(aprovadas)} imagens aprovadas e salvas em: {OUTPUT_DIR}")

# ==========================================
# ETAPA 5: EXIBIR AS IMAGENS GRAVADAS
# ==========================================
print("Exibindo as imagens processadas:")

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, img_path in enumerate(aprovadas):
    if i < 10:
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(f"Imagem {i + 1}", fontsize=10)

for j in range(len(aprovadas), 10):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


Análise de Segurança baseada no método STRIDE (com exportação PDF e DOCX)

> Considera **todos os `.png`** em `diagramas_aprovados/`, independentemente da seleção automática anterior.

In [ ]:
# ==========================================
# ETAPA 6: ANÁLISE DE SEGURANÇA (STRIDE) + PDF + DOCX
# ==========================================
import os
import json
import base64
import html
from datetime import datetime
from pathlib import Path

from google.colab import userdata

!pip install -q openai reportlab python-docx

from openai import OpenAI
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt
from docx.oxml.ns import qn
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    Image as RLImage,
    PageBreak,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
)

# Fontes padrão PDF (Core 14) — compatíveis em qualquer leitor, sem instalação
FONT_REGULAR = "Helvetica"
FONT_BOLD = "Helvetica-Bold"
FONT_MONO = "Courier"

print("Configurando a API da OpenAI...")
try:
    api_key = userdata.get("API_KEY_CHAT_GPT")
    client = OpenAI(api_key=api_key)
except Exception as e:
    print("ERRO: Verifique se o secret 'API_KEY_CHAT_GPT' foi criado corretamente.")
    raise e


def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def analisar_arquitetura_stride(image_path):
    base64_image = encode_image_to_base64(image_path)

    prompt_seguranca = """
    Você é um Arquiteto de Cloud e Especialista em Segurança da Informação.
    Analise o diagrama de arquitetura de sistemas fornecido na imagem.

    Aplique a metodologia STRIDE e forneça um relatório estruturado avaliando as potenciais vulnerabilidades. Para cada letra do STRIDE, forneça:
    1. Ameaça/Vulnerabilidade potencial identificada (baseada nos componentes visíveis ou na ausência deles).
    2. Contramedidas de Hardening recomendadas.

    As categorias são:
    - S: Spoofing (Falsificação de Identidade)
    - T: Tampering (Modificação de Dados)
    - R: Repudiation (Repúdio)
    - I: Information Disclosure (Divulgação de Informação)
    - D: Denial of Service (Negação de Serviço)
    - E: Elevation of Privilege (Elevação de Privilégio)

    Seja técnico, direto e foque em práticas modernas de segurança em nuvem (Zero Trust, IAM, Criptografia, WAF, etc.).
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_seguranca},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}",
                                "detail": "high",
                            },
                        },
                    ],
                }
            ],
            max_tokens=1500,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erro ao processar a imagem: {e}"


def _escape_pdf_text(text):
    return html.escape(text or "").replace("\n", "<br/>")


def _pdf_styles():
    return {
        "title": ParagraphStyle(
            "Title",
            fontName=FONT_BOLD,
            fontSize=18,
            leading=22,
            alignment=TA_CENTER,
            spaceAfter=12,
        ),
        "subtitle": ParagraphStyle(
            "Subtitle",
            fontName=FONT_REGULAR,
            fontSize=11,
            leading=14,
            alignment=TA_CENTER,
            textColor=colors.grey,
            spaceAfter=24,
        ),
        "heading": ParagraphStyle(
            "Heading",
            fontName=FONT_BOLD,
            fontSize=13,
            leading=16,
            spaceBefore=12,
            spaceAfter=8,
        ),
        "body": ParagraphStyle(
            "Body",
            fontName=FONT_REGULAR,
            fontSize=10,
            leading=14,
            alignment=TA_JUSTIFY,
            spaceAfter=6,
        ),
        "mono": ParagraphStyle(
            "Mono",
            fontName=FONT_MONO,
            fontSize=9,
            leading=12,
            spaceAfter=6,
        ),
    }


def gerar_pdf_relatorio(resultados, pdf_path):
    """Gera PDF consolidado com análises STRIDE usando fontes padrão PDF."""
    styles = _pdf_styles()
    doc = SimpleDocTemplate(
        str(pdf_path),
        pagesize=A4,
        rightMargin=2 * cm,
        leftMargin=2 * cm,
        topMargin=2 * cm,
        bottomMargin=2 * cm,
        title="Relatório STRIDE - Diagramas de Arquitetura",
        author="TechChallenger",
    )

    story = []
    agora = datetime.now().strftime("%d/%m/%Y %H:%M")
    story.append(Paragraph("Relatório de Análise STRIDE", styles["title"]))
    story.append(
        Paragraph(
            f"Diagramas de arquitetura validados | Gerado em {agora}",
            styles["subtitle"],
        )
    )
    story.append(Spacer(1, 0.5 * cm))

    for idx, item in enumerate(resultados, start=1):
        nome = item["arquivo"]
        analise = item["analise"]
        img_path = item["caminho"]

        story.append(Paragraph(f"Diagrama {idx}: {_escape_pdf_text(nome)}", styles["heading"]))

        if os.path.exists(img_path):
            try:
                img = RLImage(img_path, width=14 * cm, height=8 * cm, kind="proportional")
                story.append(img)
                story.append(Spacer(1, 0.3 * cm))
            except Exception:
                story.append(
                    Paragraph(
                        f"[Miniatura indisponível: {_escape_pdf_text(img_path)}]",
                        styles["mono"],
                    )
                )

        for paragrafo in (analise or "").split("\n\n"):
            paragrafo = paragrafo.strip()
            if paragrafo:
                story.append(Paragraph(_escape_pdf_text(paragrafo), styles["body"]))

        if idx < len(resultados):
            story.append(PageBreak())

    doc.build(story)
    return pdf_path


def _configurar_fonte_docx(documento, nome_fonte="Arial", tamanho=11):
    """Define fonte padrão do DOCX (Arial — disponível nativamente no Word)."""
    estilo = documento.styles["Normal"]
    fonte = estilo.font
    fonte.name = nome_fonte
    fonte.size = Pt(tamanho)
    estilo.element.rPr.rFonts.set(qn("w:eastAsia"), nome_fonte)


def gerar_docx_relatorio(resultados, docx_path):
    """Gera relatório DOCX consolidado com análises STRIDE."""
    doc = Document()
    _configurar_fonte_docx(doc)

    titulo = doc.add_heading("Relatório de Análise STRIDE", level=0)
    titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

    agora = datetime.now().strftime("%d/%m/%Y %H:%M")
    subtitulo = doc.add_paragraph(
        f"Diagramas de arquitetura validados | Gerado em {agora}"
    )
    subtitulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

    for idx, item in enumerate(resultados, start=1):
        nome = item["arquivo"]
        analise = item["analise"]
        img_path = item["caminho"]

        doc.add_heading(f"Diagrama {idx}: {nome}", level=1)

        if os.path.exists(img_path):
            try:
                doc.add_picture(img_path, width=Inches(5.5))
            except Exception:
                doc.add_paragraph(f"[Imagem indisponível: {img_path}]")

        for paragrafo in (analise or "").split("\n\n"):
            paragrafo = paragrafo.strip()
            if paragrafo:
                p = doc.add_paragraph(paragrafo)
                p.paragraph_format.space_after = Pt(6)

        if idx < len(resultados):
            doc.add_page_break()

    doc.save(str(docx_path))
    return docx_path


# ==========================================
# EXECUTANDO A ANÁLISE E GERANDO PDF/DOCX NO DRIVE
# ==========================================

def listar_png_diagramas_aprovados(diretorio):
    """Lista todos os arquivos .png da pasta diagramas_aprovados (seleção automática ou manual)."""
    pasta = Path(diretorio)
    if not pasta.exists():
        raise FileNotFoundError(f"Pasta não encontrada: {pasta}")

    imagens = sorted(
        p for p in pasta.iterdir()
        if p.is_file() and p.suffix.lower() == ".png"
    )
    return [str(p) for p in imagens]


OUTPUT_DIR = str(OUTPUT_DIR)
REPORTS_DIR = str(REPORTS_DIR)

diagramas_para_analise = listar_png_diagramas_aprovados(OUTPUT_DIR)

print("\nIniciando a avaliação STRIDE para os diagramas em diagramas_aprovados...\n")
print(f"Pasta: {OUTPUT_DIR}")
print(f"Arquivos .png encontrados: {len(diagramas_para_analise)}")

if not diagramas_para_analise:
    raise FileNotFoundError(
        f"Nenhum arquivo .png encontrado em {OUTPUT_DIR}. "
        "Adicione os diagramas manualmente ou execute a etapa de filtragem."
    )

for nome in [os.path.basename(p) for p in diagramas_para_analise]:
    print(f"  - {nome}")

resultados_stride = []

for i, img_path in enumerate(diagramas_para_analise):
    nome_arquivo = os.path.basename(img_path)
    print(f"=== Analisando Imagem {i + 1}: {nome_arquivo} ===")

    resultado_stride = analisar_arquitetura_stride(img_path)
    resultados_stride.append(
        {
            "arquivo": nome_arquivo,
            "caminho": img_path,
            "analise": resultado_stride,
        }
    )

    print(resultado_stride)
    print("\n" + "=" * 80 + "\n")

# Salva JSON com os resultados no Google Drive
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = Path(REPORTS_DIR) / f"analise_stride_{timestamp}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(resultados_stride, f, ensure_ascii=False, indent=2)

# Gera PDF e DOCX consolidados no Google Drive
pdf_path = Path(REPORTS_DIR) / f"relatorio_stride_{timestamp}.pdf"
docx_path = Path(REPORTS_DIR) / f"relatorio_stride_{timestamp}.docx"
gerar_pdf_relatorio(resultados_stride, pdf_path)
gerar_docx_relatorio(resultados_stride, docx_path)

print("Análise de segurança concluída!")
print(f"Resultados JSON : {json_path}")
print(f"Relatório PDF   : {pdf_path}")
print(f"Relatório DOCX  : {docx_path}")
print("\nArquivos gravados no Google Drive em:", REPORTS_DIR)
